# Notebook 0 — Environment Setup & Domain Introduction
**Duration:** 3.5 hours  
**Audience:** Undergraduate CS students (intermediate Python, introductory electrochemistry)  
**Goal:** Get a working Jupyter/GPU session on NERSC Perlmutter, clone the Fe-Redox-GNN repository, and understand the electrochemistry and computational motivation behind the project.

What you'll learn today
- How to start a GPU-enabled Jupyter session at NERSC (Perlmutter)
- How to create a reproducible Python environment for the workshop
- Basic electrochemistry of Fe²⁺ / Fe³⁺ redox chemistry and the Nernst equation
- Why DFT is expensive and how ML (especially GNNs) can accelerate materials screening
- Repository layout and where to find the dataset used in the project

Mentor checkpoints are included throughout. Stop at each checkpoint and confirm with your mentor before proceeding.

---

## Part 1 — HPC Environment Setup at NERSC (Perlmutter)

Perlmutter is NERSC's flagship GPU system equipped with NVIDIA A100 GPUs. During this workshop you'll run notebooks on GPU-enabled nodes using NERSC's JupyterHub (recommended) or via an interactive SLURM allocation if you prefer the terminal.

---



## 1.1 Logging into NERSC (JupyterHub + SSH alternative)

Open your browser and go to:

https://jupyter.nersc.gov

Log in with your NERSC username and OTP (MFA). When launching a server, choose a GPU-enabled configuration:

| Setting | Value |
|---|---|
| Node Type | GPU |
| Number of GPUs | 1 |
| QOS | interactive or shared (follow instructor guidance) |
| Time Limit | 4:00:00 |
| Account / Project | (your workshop allocation code) |

Behind the scenes JupyterHub submits a SLURM allocation for you.

SSH + interactive SLURM (alternative)
```bash
# SSH to Perlmutter (example)
ssh <your_username>@perlmutter-p1.nersc.gov

# Request an interactive GPU session
salloc --nodes=1 --qos=interactive --time=04:00:00 --constraint=gpu --gpus=1 --account=<your_project>
```

---

## 1.2 Clone the repository

Open a Terminal in JupyterHub (File → New → Terminal) and run:

```bash
# Navigate to your scratch directory (faster I/O for large datasets)
cd $SCRATCH

# Clone the Fe-Redox-GNN repository
git clone https://github.com/alvarovm/Fe-Redox-GNN.git

# Enter the project directory
cd Fe-Redox-GNN

# Inspect the repository structure
ls -la
```

If `tree` is available you can use `tree -L 2`, otherwise list files with `find`:
```bash
find . -type f | head -40
```

---

## 1.3 Setting up the Python environment (conda + pip)

We recommend a dedicated conda environment for reproducibility. Example commands (adapt for your site policies):

```bash
# Load the conda/python module (NERSC)
module load python

# Create a new conda environment for the workshop
conda create -n fe-redox python=3.10 -y

# Activate the environment
conda activate fe-redox

# Install core data science packages
pip install numpy pandas matplotlib seaborn scikit-learn jupyterlab ipykernel

# Install PyTorch with CUDA support (Perlmutter uses CUDA 11.8 / PyTorch 2.x builds)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install PyTorch Geometric and dependencies (choose appropriate wheel for your torch/cuda versions)
pip install torch-geometric
pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

# Chemistry-specific packages (optional)
pip install ase rdkit-pypi

# Register the environment as a kernel for Jupyter
python -m ipykernel install --user --name fe-redox --display-name "Python (Fe-Redox)"
```

After installing the kernel, refresh your JupyterHub page and select "Python (Fe-Redox)" when creating notebooks.

---

## 1.4 Verification: Test your setup

Create a new notebook using the "Python (Fe-Redox)" kernel and run the verification cell below to confirm package versions and GPU availability.

---




In [ ]:

# Environment verification: run this cell in your Notebook 0 to confirm setup
import sys
import os

print(f"Python version: {sys.version.splitlines()[0]}")
print(f"Python executable: {sys.executable}")
print("=" * 60)

# Core data science stack
import numpy as np
print(f"NumPy version: {np.__version__}")

import pandas as pd
print(f"Pandas version: {pd.__version__}")

import matplotlib
print(f"Matplotlib version: {matplotlib.__version__}")

import seaborn as sns
print(f"Seaborn version: {sns.__version__}")

import sklearn
print(f"Scikit-learn version: {sklearn.__version__}")

# Deep learning stack
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    try:
        print(f"GPU device: {torch.cuda.get_device_name(0)}")
    except Exception:
        print("GPU device: (couldn't query device name)")
    print(f"CUDA version: {torch.version.cuda}")

# PyTorch Geometric (import may be delayed if wheels are not installed correctly)
try:
    import torch_geometric
    print(f"PyTorch Geometric version: {torch_geometric.__version__}")
except Exception as e:
    print("Could not import torch_geometric:", e)

print("=" * 60)
print("If the above show expected versions and CUDA availability=True, you're ready to proceed.")



---

**Mentor checkpoint 1**

Before proceeding, confirm with your mentor:
- You can access NERSC JupyterHub
- The repository is cloned successfully under `$SCRATCH/Fe-Redox-GNN`
- All packages import without errors
- PyTorch detects the GPU

Proceed only after confirmation.

---




## Part 2 — Domain Science: The Electrochemistry of Iron (90 minutes)

Iron is abundant, inexpensive, and non-toxic compared with many other transition metals. Its multiple accessible oxidation states and wide tunability via ligands make iron attractive for grid-scale energy storage and sustainable metal extraction.

Applications we care about:
- Grid-scale energy storage: iron-based redox flow batteries (RFBs)
- Sustainable metal extraction (electrowinning / hydrometallurgy)
- Fundamental research in redox catalysis and coordination chemistry

---

### 2.1 Fundamentals of Redox Chemistry (Fe²⁺ / Fe³⁺)

Redox reactions are electron-transfer processes.

Oxidation (loss of electrons):

$$\mathrm{Fe}^{2+} \longrightarrow \mathrm{Fe}^{3+} + e^{-}$$

Reduction (gain of electrons):

$$\mathrm{Fe}^{3+} + e^{-} \longrightarrow \mathrm{Fe}^{2+}$$

Combined half-reaction (reversible):

$$\mathrm{Fe}^{2+} \rightleftharpoons \mathrm{Fe}^{3+} + e^{-}$$

The standard reduction potential quantifies the thermodynamic tendency to be reduced:

$$E^{0}(\mathrm{Fe}^{3+}/\mathrm{Fe}^{2+}) = +0.77\ \text{V vs. SHE}$$

where SHE = Standard Hydrogen Electrode.

---

### 2.2 The Nernst Equation

Under non-standard conditions the cell potential is given by the Nernst equation:

$$
E = E^{0} - \frac{RT}{nF}\ln Q
$$
where
- $E$ is the cell potential (V),
- $E^{0}$ is the standard reduction potential (V),
- $R = 8.314\ \mathrm{J\,mol^{-1}\,K^{-1}}$,
- $T$ is temperature (K),
- $n$ is number of electrons transferred,
- $F = 96485\ \mathrm{C\,mol^{-1}}$ (Faraday constant),
- $Q$ is the reaction quotient.

At $T = 298.15\ \mathrm{K}$ (25 °C) this becomes:

$$
E = E^{0} - \frac{0.0592}{n}\log_{10} Q \quad (\text{V, at } 298.15\ \text{K})
$$

For the Fe³⁺/Fe²⁺ couple, $n = 1$.

---

### 2.3 Redox Flow Batteries (All-Iron RFB)

RFBs store energy in dissolved redox-active species in tanks; power is provided by the electrochemical cell stack and capacity scales with electrolyte volume.

All-iron RFB half-reactions and standard potentials:

| Half-cell (role) | Reaction | $E^{0}$ (V vs. SHE) |
|---|---:|---:|
| Positive (cathode) | $\mathrm{Fe}^{3+} + e^{-} \rightarrow \mathrm{Fe}^{2+}$ | +0.77 |
| Negative (anode) | $\mathrm{Fe}^{2+} + 2e^{-} \rightarrow \mathrm{Fe}^{0}$ | −0.44 |

Cell open-circuit voltage:

$$
E^{0}_{\text{cell}} = E^{0}_{\text{cathode}} - E^{0}_{\text{anode}} = 0.77 - (-0.44) = 1.21\ \text{V}
$$

Note on full balanced reaction: multiply the cathode half-reaction by 2 to balance electrons:

$$
2\,\mathrm{Fe}^{3+} + \mathrm{Fe}^{0} \rightarrow 3\,\mathrm{Fe}^{2+}
$$

Advantages of iron RFBs
- Low cost and abundant materials
- Long cycle life
- Aqueous, non-flammable electrolytes

---



### 2.4 Ligand Environment and Tunability of Redox Potentials

Ligands (the atoms/molecules surrounding Fe) dramatically influence the redox potential by stabilizing different oxidation states. Structural features that matter:
- Coordination number (4, 5, 6, ...)
- Bond lengths (Fe–O, Fe–N, Fe–S)
- Ligand field strength (spectrochemical series)
- Geometry (octahedral, tetrahedral, square planar)

Selected example complexes and their standard reduction potentials:

| Complex (notation) | $E^{0}$ (V vs. SHE) |
|---|---:|
| $[\mathrm{Fe(H_2O)_6}]^{3+/2+}$ (aquo) | +0.77 |
| $[\mathrm{Fe(CN)_6}]^{3-/4-}$ (cyano) | +0.36 |
| $[\mathrm{Fe(phen)_3}]^{3+/2+}$ (phenanthroline) | +1.06 |
| $[\mathrm{Fe(ox)_3}]^{3-/4-}$ (oxalate) | +0.02 |

This tunability is why computational screening is valuable: we can predict how ligand changes shift potentials without synthesizing each complex.

---

### 2.5 Visualizing ligand-induced shifts (example)

Below is a simple visualization of typical standard reduction potentials for selected iron complexes. Paste the following code into a code cell and run it to produce a horizontal bar chart.


In [ ]:

# Visualization: Iron complexes and their standard reduction potentials
import matplotlib.pyplot as plt
import numpy as np

# Data: Iron complexes and their redox potentials (V vs. SHE)
complexes = {
    r'$[\mathrm{Fe(H_2O)_6}]^{3+/2+}$': 0.77,
    r'$[\mathrm{Fe(CN)_6}]^{3-/4-}$': 0.36,
    r'$[\mathrm{Fe(phen)_3}]^{3+/2+}$': 1.06,
    r'$[\mathrm{Fe(ox)_3}]^{3-/4-}$': 0.02,
    r'$[\mathrm{Fe(edta)}]^{-/2-}$': 0.12,
    r'$[\mathrm{Fe(bipy)_3}]^{3+/2+}$': 1.03,
}

names = list(complexes.keys())
potentials = list(complexes.values())

# Create a horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(names)))
bars = ax.barh(names, potentials, color=colors, edgecolor='black', linewidth=0.8)

# Add value labels on bars
for bar, val in zip(bars, potentials):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f} V', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Standard Reduction Potential (V vs. SHE)', fontsize=13)
ax.set_title('Effect of Ligand Environment on Fe3+/Fe2+ Redox Potential',
             fontsize=14, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlim(-0.2, 1.3)
plt.tight_layout()
plt.savefig('iron_redox_potentials.png', dpi=150, bbox_inches='tight')
plt.show()

print("Notice how the same Fe2+/Fe3+ couple spans over ~1 V depending on the ligand.")





## Part 3 — The Computational Challenge: Why Machine Learning?

### 3.1 DFT: The gold standard
Density Functional Theory (DFT) computes electronic energies and can be used to obtain redox free energies by computing total energies of different oxidation states.

Gibbs free energy difference for the redox event:

$$
\Delta G_{\text{redox}} = G(\text{Fe}^{3+}\text{-complex}) - G(\text{Fe}^{2+}\text{-complex})
$$

The standard reduction potential follows from:
---
$$
E^{0} = -\frac{\Delta G_{\text{redox}}}{nF}
$$

(Here $\Delta G_{\text{redox}}$ should be expressed in J/mol if you use $F$ in C/mol. In computational workflows one often works with energies in eV per electron; be careful to use consistent units when applying the formula.)

Typical cost/accuracy tradeoffs (rough example):
| Method | Time per molecule | Typical accuracy (approx) |
|---|---:|---:|
| DFT (e.g., hybrid functional) | hours–days | ~0.1 V |
| Classical ML (Random Forest on descriptors) | ~ms | ~0.2–0.4 V |
| Graph Neural Network (GNN) | ~10 ms (inference) | ~0.1–0.2 V (with good data) |

A single DFT calculation for a medium-sized iron complex can take many hours. Screening thousands of candidates is expensive — ML surrogates trained on DFT labels speed up screening by several orders of magnitude.

---

### 3.2 The Machine Learning Workflow (project overview)

High-level workflow for the Fe-Redox-GNN project:

```
Generate iron complex structures  ──▶  Run DFT (labels)  ──▶  Extract redox potentials
                                                     │
                                                     ▼
                                          Represent molecules as graphs
                                                     │
                                                     ▼
                                          Train GNN on graph data
                                                     │
                                                     ▼
                                      Predict redox potentials for new molecules
```

We will follow this workflow in the workshop: find the dataset in the repo, build features and graphs, train baseline ML models, and then train a GNN on GPU.

---

### 3.3 Why Graph Neural Networks?

Traditional ML depends on hand-crafted descriptors (bond lengths, angles, coordination numbers). Handcrafting can miss important information and is often system-specific.

GNNs operate on graphs directly:
- Nodes = atoms (features: atomic number, formal charge, etc.)
- Edges = bonds or neighbor distances (features: bond length, bond order)
- Message passing aggregates neighbor information and learns hierarchical representations

This representation is flexible and transferable across different chemistries and sizes.

---



### 3.4 Exploring the repository structure

We will programmatically inspect the repository to find data files, scripts, and notebooks. The example below assumes you cloned the repo at `$SCRATCH/Fe-Redox-GNN`. If you cloned elsewhere, change `repo_path` accordingly.

```python


In [ ]:

# Explore repository tree (adjust repo_path if needed)
import os

repo_path = os.path.expandvars('$SCRATCH/Fe-Redox-GNN')
if not os.path.exists(repo_path):
    print("Repository path not found:", repo_path)
    print("If you cloned elsewhere, update repo_path accordingly.")
else:
    print("Fe-Redox-GNN Repository Structure")
    print("=" * 60)
    for root, dirs, files in os.walk(repo_path):
        # Skip hidden directories (.git)
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        level = root.replace(repo_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:10]:  # Limit to first 10 files per directory
            try:
                size = os.path.getsize(os.path.join(root, file))
            except OSError:
                size = 0
            size_str = f"{size / 1024:.1f} KB" if size > 1024 else f"{size} B"
            print(f'{subindent}{file} ({size_str})')
        if len(files) > 10:
            print(f'{subindent}... and {len(files) - 10} more files')

# Read README.md or search for data files
import os

repo_path = os.path.expandvars('$SCRATCH/Fe-Redox-GNN')
readme_path = os.path.join(repo_path, 'README.md')

if os.path.exists(readme_path):
    with open(readme_path, 'r', encoding='utf-8') as f:
        print("README.md (first 3000 chars):")
        print("=" * 60)
        print(f.read()[:3000])
else:
    print("README.md not found. Searching for data files...")
    for root, dirs, files in os.walk(repo_path):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for f in files:
            if f.lower().endswith(('.csv', '.json', '.pkl', '.npz', '.xyz', '.npy', '.pt', '.dat', '.sdf')):
                filepath = os.path.join(root, f)
                try:
                    size = os.path.getsize(filepath) / 1024
                except OSError:
                    size = 0
                print(f"  {os.path.relpath(filepath, repo_path)} ({size:.1f} KB)")





---

### 3.5 Gibbs Free Energy and Redox Potentials

Thermodynamically, the redox free energy difference between two oxidation states gives the standard reduction potential.

Gibbs free energy difference for the redox reaction:

$$
\Delta G_{\text{redox}} = G(\text{Fe}^{3+}\text{-complex}) - G(\text{Fe}^{2+}\text{-complex})
$$

Standard reduction potential:

$$
E^{0} = -\frac{\Delta G_{\text{redox}}}{nF}
$$

Notes on units:
- If $\Delta G_{\text{redox}}$ is in J/mol, use $F = 96485\ \mathrm{C/mol}$ and the formula yields volts directly.
- In computational chemistry it is common to work in eV per electron. If $\Delta G$ is given in eV per electron, then numerically $-\Delta G/n$ gives the potential in volts relative to the absolute vacuum scale; to convert to the SHE scale subtract the absolute SHE potential (commonly ~4.28 V, but the exact value depends on the convention). Always be explicit about units.



In [ ]:

# Compute redox potentials and plot Nernst behavior
import numpy as np
import matplotlib.pyplot as plt

# Physical constants
R = 8.314          # J/(mol·K) - Gas constant
F = 96485.0        # C/mol - Faraday's constant
T = 298.15         # K - Standard temperature (25°C)

def compute_redox_potential(delta_G_eV_per_electron, n_electrons=1, E_ref_SHE=4.28):
    """
    Compute the standard redox potential in V vs. SHE from a Gibbs free energy.

    Parameters
    ----------
    delta_G_eV_per_electron : float
        Gibbs free energy change for the redox reaction expressed as eV per electron.
        (This is a common unit in computational chemistry outputs.)
    n_electrons : int
        Number of electrons transferred.
    E_ref_SHE : float
        Absolute potential of the Standard Hydrogen Electrode in V (approx. 4.28 V).
        Subtracting this converts an absolute potential to the conventional SHE scale.

    Returns
    -------
    E0_vs_SHE : float
        Standard reduction potential in V vs. SHE.
    """
    # If delta_G is in eV/electron then -delta_G/n (eV) ≈ potential in volts on an absolute scale.
    E0_abs = -delta_G_eV_per_electron / n_electrons  # in V (approx, when using eV per electron)
    E0_vs_SHE = E0_abs - E_ref_SHE
    return E0_vs_SHE

def nernst_potential(E0, T, n, Q):
    """
    Compute the cell potential using the Nernst equation.
    E = E0 - (RT)/(nF) * ln(Q)
    """
    E = E0 - (R * T) / (n * F) * np.log(Q)
    return E

# Example: Fe3+/Fe2+ couple
E0_Fe = 0.77  # V vs. SHE

# Compute potential at different concentration ratios
ratios = np.logspace(-3, 3, 200)  # [Fe2+]/[Fe3+] from 0.001 to 1000
E_values = [nernst_potential(E0_Fe, T, n=1, Q=q) for q in ratios]

# Plot the Nernst equation
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(np.log10(ratios), E_values, 'b-', linewidth=2.5)
ax.axhline(y=E0_Fe, color='red', linestyle='--', linewidth=1.5, label=f'$E^0$ = {E0_Fe} V')
ax.axvline(x=0, color='gray', linestyle=':', linewidth=1)

ax.set_xlabel(r'$\log_{10}([\mathrm{Fe}^{2+}]/[\mathrm{Fe}^{3+}])$', fontsize=13)
ax.set_ylabel('Cell Potential $E$ (V vs. SHE)', fontsize=13)
ax.set_title('Nernst Equation: Fe3+/Fe2+ Couple at 25°C', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('nernst_equation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"At equal concentrations (Q=1): E = {nernst_potential(E0_Fe, T, 1, 1):.3f} V")
print(f"At [Fe2+]/[Fe3+] = 10:       E = {nernst_potential(E0_Fe, T, 1, 10):.3f} V")
print(f"At [Fe2+]/[Fe3+] = 0.1:      E = {nernst_potential(E0_Fe, T, 1, 0.1):.3f} V")



---

## Exercises & Discussion Questions

Work through the programming exercises below and discuss conceptual questions with your neighbor. Stop at the mentor checkpoint when complete.

---

### Exercise 0.1 — Nernst Equation Exploration

Tasks:
1. Compute the potential of the Fe3+/Fe2+ couple at 80°C with $[\mathrm{Fe}^{2+}]/[\mathrm{Fe}^{3+}] = 0.01$.
2. Solve for the concentration ratio $[\mathrm{Fe}^{2+}]/[\mathrm{Fe}^{3+}]$ that yields $E = 0.50\ \text{V}$ at 25°C.
3. Plot the Nernst equation at three temperatures (25°C, 50°C, 80°C) on the same graph.

Below is starter code. Complete Q2 and Q3 where indicated.



In [ ]:

# Exercise 0.1 starter code
import numpy as np

# Constants (re-used)
R = 8.314          # J/(mol·K)
F = 96485.0        # C/mol

# Given values
E0_Fe = 0.77  # V vs SHE (standard)
# Q1: potential at 80°C and Q = 0.01
T_hot = 273.15 + 80  # K
Q_ratio = 0.01
E_hot = nernst_potential(E0_Fe, T_hot, n=1, Q=Q_ratio)
print(f"Q1: E at 80°C, Q=0.01: {E_hot:.4f} V")

# Q2: find Q such that E = 0.5 V at 25°C
T_std = 298.15
E_target = 0.5

# Solve for Q from E = E0 - (RT)/(nF) * ln Q
# ln Q = (E0 - E) * nF / (RT)
# Q = exp( ln Q )
ln_Q = (E0_Fe - E_target) * 1 * F / (R * T_std)
Q_target = np.exp(ln_Q)
print(f"Q2: [Fe^2+]/[Fe^3+] ratio for E=0.5 V at 25°C: {Q_target:.4e}")

# Q3: Plot Nernst at 3 temperatures
import matplotlib.pyplot as plt

temps_C = [25, 50, 80]
ratios = np.logspace(-3, 3, 200)

fig, ax = plt.subplots(figsize=(8, 5))
for Tc in temps_C:
    T_k = 273.15 + Tc
    E_vals = [nernst_potential(E0_Fe, T_k, n=1, Q=q) for q in ratios]
    ax.plot(np.log10(ratios), E_vals, label=f'{Tc} °C')

ax.set_xlabel(r'$\log_{10}([\mathrm{Fe}^{2+}]/[\mathrm{Fe}^{3+}])$')
ax.set_ylabel('E (V vs. SHE)')
ax.set_title('Nernst Equation at different temperatures')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()




---

### Exercise 0.2 — Conceptual Questions (write answers in a Markdown cell)

1. Why is iron preferred over vanadium for large-scale redox flow batteries, despite vanadium RFBs being more commercially mature?  
2. If a ligand stabilizes Fe3+ more than Fe2+, will the redox potential shift to more positive or more negative values? Explain using Gibbs free energy relationships.  
3. Why can't we simply use a lookup table of known redox potentials instead of training a machine learning model? What advantages does ML provide?  
4. What is the key limitation of DFT that motivates the use of machine learning surrogates?

Write concise answers and be prepared to discuss them.

---

### Discussion Question

"If we can predict redox potentials with ML, does that mean we no longer need experimental electrochemistry?"

Discuss with your neighbor for 5 minutes and then share with the class. Consider limitations of models, data quality, and the need for experimental validation.





---

**Mentor checkpoint 2**

Before leaving today, verify with your mentor:
- You understand how ligand environment influences redox potential
- You can explain why GNNs are well-suited for molecular property prediction
- You completed Exercise 0.1 and plotted the Nernst equation
- Your environment (kernel, imports, GPU) is fully set up for Day 2

Proceed only after confirmation.

---




## Summary: What we covered today

| Topic | Key takeaway |
|---|---|
| NERSC Setup | Use JupyterHub to launch GPU-backed notebooks; SLURM allocates nodes |
| Iron Redox Chemistry | $ \mathrm{Fe}^{2+} \rightleftharpoons \mathrm{Fe}^{3+} + e^{-}$, ligand tunability |
| Nernst Equation | $E = E^{0} - \dfrac{RT}{nF}\ln Q$ (or $E = E^{0} - \dfrac{0.0592}{n}\log_{10}Q$ at 25°C) |
| Redox Flow Batteries | Capacity determined by electrolyte volume; cell voltage from half-cell potentials |
| DFT vs ML | DFT is accurate but slow; ML (esp. GNNs) can approximate DFT-quality labels for screening |




---


## Preparation for Day 2
Before tomorrow:
- Ensure your `fe-redox` conda environment is activated and working
- Explore the cloned repo and identify the data files (CSV/NPY/PKL) used by the project
- Brush up on Pandas basics (Pandas 10-minute tutorial: https://pandas.pydata.org/docs/getting_started/10min.html)

We will load and clean the dataset, visualize structure-property relationships, and prepare features for ML models on Day 2.
